# ARIMA / SARIMA — Store Sales Forecasting

ARIMA(p, d, q) კლასიკური სტატისტიკური მოდელია:
- **AR(p)** — მიმდინარე მნიშვნელობა წარსული მნიშვნელობების წრფივი კომბინაციაა;
- **I(d)** — differencing (რიგის d), ხდის სერიას stationary (ტრენდის მოშორება);
- **MA(q)** — წარსული შეცდომების წრფივი კომბინაცია.

**SARIMA(p,d,q)(P,D,Q,s)** ამატებს სეზონურ ნაწილს პერიოდით s (აქ 52 კვირა).

per-series SARIMA s=52-ით 3300 სერიაზე ძალიან ნელია (და დავალებაც ამბობს, დიდი დრო
ნუ დაიხარჯებაო). ამიტომ ორ ნაწილად ვაკეთებ:
1. **seasonal-naive** ყველა სერიაზე — იაფი, სრულ holdout-ზე;
2. **SARIMA აგრეგირებულ სერიაზე** — თეორიის/დიაგნოსტიკის საჩვენებლად (AIC).

### 1. ბიბლიოთეკები + მონაცემები

In [ ]:
import sys
import os
import warnings

sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")
os.environ.setdefault("WANDB_SILENT", "true")

import numpy as np
import statsmodels.api as sm

from src.classical_models import build_seasonal_naive_pipeline
from src.data import load_raw
from src.metrics import wmae
from src.pipeline import RAW_COLS
from src.validation import time_holdout_split
from src.wandb_utils import init_run, log_pipeline

train = load_raw("data").train
print("train:", train.shape)

### 2. Seasonal-naive (მთელ holdout-ზე)

იდეა მარტივია: მომავალი კვირის პროგნოზი = **იმავე სერიის მნიშვნელობა 52 კვირის წინ**.
ეს არის SARIMA-ს კერძო შემთხვევა `(0,1,0)(0,1,0,52)` — მხოლოდ სეზონური differencing.

In [ ]:
tr, val = time_holdout_split(train, n_val_weeks=12)
val = val.reset_index(drop=True)

run = init_run(group="ARIMA_Training", job_type="experiment", name="ARIMA_SeasonalNaive",
               config={"model": "seasonal_naive", "season_weeks": 52})

pipe = build_seasonal_naive_pipeline()
pipe.fit(tr[RAW_COLS], tr["Weekly_Sales"])

pred = pipe.predict(val[RAW_COLS])
seasonal_naive_wmae = wmae(val["Weekly_Sales"], pred, val["IsHoliday"])

run.summary["holdout_wmae"] = seasonal_naive_wmae
run.summary["wmae_val"] = seasonal_naive_wmae
run.finish()

print("seasonal-naive WMAE:", round(seasonal_naive_wmae, 2))

### 3. SARIMA აგრეგირებულ სერიაზე

ცალკეული სერიების ჯამზე (მთელი ქსელის კვირეული გაყიდვა) რამდენიმე order-ს ვამოწმებ და
AIC-ით ვადარებ. აქ ჩანს კარგად, რომ სეზონური differencing (s=52) საჭიროა.

In [ ]:
# ჯამური კვირეული სერია
agg = train.groupby("Date")["Weekly_Sales"].sum().sort_index()
holiday = train.groupby("Date")["IsHoliday"].max().sort_index()

cutoff = agg.index[-12]
agg_tr = agg[agg.index < cutoff]
agg_val = agg[agg.index >= cutoff]
holiday_val = holiday[holiday.index >= cutoff]

print("აგრეგატი train:", len(agg_tr), "| val:", len(agg_val))

In [ ]:
ORDERS = [
    {"name": "SARIMA_agg_111_011", "order": (1, 1, 1), "seasonal_order": (0, 1, 1, 52)},
    {"name": "SARIMA_agg_011_011", "order": (0, 1, 1), "seasonal_order": (0, 1, 1, 52)},
    {"name": "SARIMA_agg_212_110", "order": (2, 1, 2), "seasonal_order": (1, 1, 0, 52)},
    {"name": "SARIMA_agg_110_011", "order": (1, 1, 0), "seasonal_order": (0, 1, 1, 52)},
]

for spec in ORDERS:
    run = init_run(group="ARIMA_Training", job_type="experiment", name=spec["name"],
                   config={"order": str(spec["order"]),
                           "seasonal_order": str(spec["seasonal_order"]),
                           "level": "aggregate"})

    model = sm.tsa.statespace.SARIMAX(agg_tr, order=spec["order"],
                                      seasonal_order=spec["seasonal_order"],
                                      enforce_stationarity=False,
                                      enforce_invertibility=False)
    fitted = model.fit(disp=False, maxiter=50)

    forecast = np.asarray(fitted.forecast(len(agg_val)))
    mape = np.mean(np.abs((agg_val.values - forecast) / agg_val.values)) * 100
    agg_wmae = wmae(agg_val.values, forecast, holiday_val.values)

    run.summary["aic"] = float(fitted.aic)
    run.summary["agg_mape_pct"] = float(mape)
    run.summary["agg_wmae"] = agg_wmae
    run.finish()

    print(spec["name"], "| AIC:", round(fitted.aic), "| MAPE:", round(mape, 2), "%")

### 4. Final — seasonal-naive მთელ train-ზე → `walmart_arima:best`

In [ ]:
run = init_run(group="ARIMA_Training", job_type="final", name="ARIMA_Final",
               config={"model": "seasonal_naive", "season_weeks": 52})

final_pipe = build_seasonal_naive_pipeline()
final_pipe.fit(train[RAW_COLS], train["Weekly_Sales"])

run.summary["holdout_wmae"] = seasonal_naive_wmae
run.summary["wmae_val"] = seasonal_naive_wmae
log_pipeline(run, final_pipe, name="walmart_arima",
             metadata={"holdout_wmae": seasonal_naive_wmae, "model": "seasonal_naive"},
             aliases=["best"])
run.finish()
print("დარეგისტრირდა: walmart_arima:best")

### შედეგები

- **seasonal-naive WMAE ≈ 1714** — მარტივი, მაგრამ XGBoost-ზე (1869) უკეთესი.
  ე.ი. წლიური სეზონურობა ისე ძლიერია, რომ „შარშან ამ კვირას“ ძალიან კარგი პროგნოზია.
- აგრეგატზე საუკეთესო order (AIC-ით) = **SARIMA(0,1,1)(0,1,1,52)**, AIC ≈ 772,
  MAPE ≈ 1.65%.